# Entity Resolution Pipeline
Runs the stages of `src/pipeline.py` in order. Each stage caches its output in `cache/`, so rerunning a cell is cheap unless noted. Details: `code/business_entity_resolution/README.md`.

In [ ]:
import sys
if "../.." not in sys.path:
    sys.path.append("../..")

import subprocess
import pandas as pd
from src import config, pipeline
from src.features import FEATURE_NAMES
from src.models import feature_importance
from src.retrieval import gpu_available

gpu_available()

## 1. Prepare
fit/holdout split, Indic lexicon, normalized parquet per source × country

In [ ]:
pipeline.prepare()

In [ ]:
lexicon = pipeline.get_lexicon()
len(lexicon), list(lexicon.items())[:10]

In [ ]:
_, pool = pipeline.load_split("train", "India", ["entity_id", "name_core", "addr_norm", "state", "has_indic"])
pool[pool["has_indic"]].sample(5, random_state=0)

In [ ]:
del pool

## 2. Candidates
FAISS reverse assignment per country × state (~90 min on first run)

In [ ]:
pipeline.all_candidates("train")
pipeline.all_candidates("test")

In [ ]:
candSizes = {c: pipeline.candidates("test", c).groupby("s1_idx").size().describe() for c in pipeline.countries("test")}
pd.DataFrame(candSizes).round(2)

## 3. Train
XGBoost, 5-fold GroupKFold (retrains on every run, ~6 min on GPU)

In [ ]:
models = pipeline.train()

In [ ]:
feature_importance(models, FEATURE_NAMES).sort_values(ascending=False).head(10)

## 4. Evaluate
gate tuned on one holdout half, benchmark on the other

In [ ]:
report = pipeline.evaluate()
report

## 5. Predict
test set → `output/`

In [ ]:
pipeline.predict_test()

In [ ]:
result = subprocess.run(
    [sys.executable, str(config.VALIDATE_SUBMISSION_SCRIPT),
     "--matching", str(config.MATCHING_RESULTS_PATH),
     "--candidate", str(config.CANDIDATE_PAIRS_PATH),
     "--test-dir", str(config.TEST_DIR)],
    capture_output=True, text=True)
print(result.stdout[-300:])

In [ ]:
testS1 = pd.read_csv(config.TEST_SOURCE1_PATH, sep="\t", usecols=["entity_id", "country"])
matches = pd.read_csv(config.MATCHING_RESULTS_PATH, sep="\t", keep_default_na=False)
matches["numMatched"] = matches["matched_entity_ids"].str.count(",").add(1).where(matches["matched_entity_ids"] != "", 0)
testS1.merge(matches, left_on="entity_id", right_on="source1_entity_id").groupby("country")["numMatched"].describe().round(2)